In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/data"  # as listed in the provided paths

train_path = f"{DATA_DIR}/train.csv"
test_path = f"{DATA_DIR}/test.csv"
sample_sub_path = f"{DATA_DIR}/sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sample_sub_path)

assert "target" in train.columns
assert "image_name" in test.columns and "image_name" in sub.columns
assert len(sub) == len(test)

JPEG_DIR_CANDIDATES = [
    f"{DATA_DIR}/jpeg",
    f"{DATA_DIR}/siim-isic-melanoma-classification/jpeg",
    "/kaggle/input/siim-isic-melanoma-classification/jpeg",
    "/kaggle/input/jpeg",
]
JPEG_DIR = next((p for p in JPEG_DIR_CANDIDATES if os.path.isdir(p)), None)
if JPEG_DIR is None:
    raise FileNotFoundError(
        "Could not find a valid JPEG directory in expected locations."
    )

TRAIN_JPEG_DIR = os.path.join(JPEG_DIR, "train")
TEST_JPEG_DIR = os.path.join(JPEG_DIR, "test")



In [2]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

tr = train.copy(deep=False)
te = test.copy(deep=False)

for c in ["sex", "anatom_site_general_challenge"]:
    tr[c] = tr[c].fillna("unknown").astype(str).str.strip().str.lower()
    te[c] = te[c].fillna("unknown").astype(str).str.strip().str.lower()

tr["age_approx"] = pd.to_numeric(tr["age_approx"], errors="coerce")
te["age_approx"] = pd.to_numeric(te["age_approx"], errors="coerce")

age_bins = [-np.inf, 20, 30, 40, 50, 60, 70, 80, np.inf]
age_labels = ["<20", "20s", "30s", "40s", "50s", "60s", "70s", "80+"]

tr["age_bin"] = pd.cut(tr["age_approx"], bins=age_bins, labels=age_labels).astype(
    object
)
te["age_bin"] = pd.cut(te["age_approx"], bins=age_bins, labels=age_labels).astype(
    object
)
tr["age_bin"] = tr["age_bin"].where(tr["age_bin"].notna(), "unknown")
te["age_bin"] = te["age_bin"].where(te["age_bin"].notna(), "unknown")

global_mean = float(tr["target"].mean())

tr["sex"] = tr["sex"].astype("category")
te["sex"] = te["sex"].astype("category")
tr["anatom_site_general_challenge"] = tr["anatom_site_general_challenge"].astype(
    "category"
)
te["anatom_site_general_challenge"] = te["anatom_site_general_challenge"].astype(
    "category"
)
tr["age_bin"] = tr["age_bin"].astype("category")
te["age_bin"] = te["age_bin"].astype("category")


def smoothed_rate(df, key, target_col="target", prior=None, m=200):
    if prior is None:
        prior = float(df[target_col].mean())
    stats = df.groupby(key, observed=True)[target_col].agg(["mean", "count"])
    stats["smooth"] = (stats["count"] * stats["mean"] + m * prior) / (
        stats["count"] + m
    )
    return stats["smooth"]


def build_te_maps(df_fit, prior, m_sex=50, m_site=120, m_age=120, m_pid=5000):
    sex_rate = smoothed_rate(df_fit, "sex", prior=prior, m=m_sex)
    site_rate = smoothed_rate(
        df_fit, "anatom_site_general_challenge", prior=prior, m=m_site
    )
    age_rate = smoothed_rate(df_fit, "age_bin", prior=prior, m=m_age)
    pid_rate = smoothed_rate(df_fit, "patient_id", prior=prior, m=m_pid)
    return sex_rate, site_rate, age_rate, pid_rate


def apply_te(df, sex_rate, site_rate, age_rate, pid_rate, fallback, disable_pid=False):
    x_sex = (
        pd.to_numeric(df["sex"].map(sex_rate), errors="coerce")
        .fillna(fallback)
        .astype(float)
    )
    x_site = (
        pd.to_numeric(
            df["anatom_site_general_challenge"].map(site_rate), errors="coerce"
        )
        .fillna(fallback)
        .astype(float)
    )
    x_age = (
        pd.to_numeric(df["age_bin"].map(age_rate), errors="coerce")
        .fillna(fallback)
        .astype(float)
    )

    if disable_pid:
        x_pid = np.full(len(df), float(fallback), dtype=np.float64)
    else:
        x_pid = (
            pd.to_numeric(df["patient_id"].map(pid_rate), errors="coerce")
            .fillna(fallback)
            .astype(float)
        )

    X = np.vstack([x_sex.values, x_site.values, x_age.values, x_pid]).T
    X = np.nan_to_num(X, nan=fallback, posinf=fallback, neginf=fallback)
    return X




In [3]:
from PIL import Image, ImageStat
from concurrent.futures import ThreadPoolExecutor

Image.MAX_IMAGE_PIXELS = None


def _img_stats_from_path_fast(path):
    """
    Returns 5 deterministic features:
    - mean_gray, std_gray from standard luminance grayscale (PIL "L") in [0,1]
    - mean_r, mean_g, mean_b in [0,1]
    """
    try:
        with Image.open(path) as im:
            try:
                im.draft("RGB", (128, 128))
            except Exception:
                pass
            im = im.convert("RGB")
            im = im.resize((128, 128), resample=Image.BILINEAR)

            st = ImageStat.Stat(im)
            mean_r, mean_g, mean_b = (
                st.mean[0] / 255.0,
                st.mean[1] / 255.0,
                st.mean[2] / 255.0,
            )

            gray_im = im.convert("L")
            gray = np.asarray(gray_im, dtype=np.float32) / 255.0
            mean_gray = float(gray.mean())
            std_gray = float(gray.std())

        return np.array([mean_gray, std_gray, mean_r, mean_g, mean_b], dtype=np.float32)
    except Exception:
        return np.array([np.nan, np.nan, np.nan, np.nan, np.nan], dtype=np.float32)


def build_image_features(
    image_names, folder, cache_path=None, max_workers=None, chunksize=2048
):
    if cache_path is not None and os.path.exists(cache_path):
        feats = np.load(cache_path)
        if feats.shape == (len(image_names), 5):
            return feats.astype(np.float32, copy=False)

    n = len(image_names)
    feats = np.zeros((n, 5), dtype=np.float32)

    if max_workers is None:
        max_workers = max(4, min(16, (os.cpu_count() or 8)))

    join = os.path.join
    paths = [join(folder, f"{name}.jpg") for name in image_names]

    def _process_range(start_end):
        s, e = start_end
        out = np.empty((e - s, 5), dtype=np.float32)
        fn = _img_stats_from_path_fast
        for j, i in enumerate(range(s, e)):
            out[j] = fn(paths[i])
        return s, out

    bounds = list(range(0, n, chunksize))
    ranges = [(s, min(s + chunksize, n)) for s in bounds]

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for s, out in ex.map(_process_range, ranges):
            feats[s : s + out.shape[0]] = out

    if cache_path is not None:
        np.save(cache_path, feats)
    return feats


tr_names = tr["image_name"].values
te_names = te["image_name"].values

tr_cache = os.path.join("/kaggle/working", "tr_img_feats_128_lumgray.npy")
te_cache = os.path.join("/kaggle/working", "te_img_feats_128_lumgray.npy")

tr_img_raw = build_image_features(
    tr_names, TRAIN_JPEG_DIR, cache_path=tr_cache, chunksize=2048
)
te_img_raw = build_image_features(
    te_names, TEST_JPEG_DIR, cache_path=te_cache, chunksize=2048
)

# Change (score-relevant, minimal): add missingness indicator flags for image stats.
# This reduces the harm of mean-imputation when an image fails to load, often improving AUC ranking.
tr_img_missing = (~np.isfinite(tr_img_raw)).astype(np.float32)
te_img_missing = (~np.isfinite(te_img_raw)).astype(np.float32)

col_means = np.nanmean(tr_img_raw, axis=0)
col_means = np.where(np.isfinite(col_means), col_means, 0.0).astype(np.float32)
tr_img = np.where(np.isfinite(tr_img_raw), tr_img_raw, col_means)
te_img = np.where(np.isfinite(te_img_raw), te_img_raw, col_means)

# Final image feature block: 5 stats + 5 missingness flags (still deterministic, very small change).
tr_img2 = np.hstack([tr_img.astype(np.float32, copy=False), tr_img_missing])
te_img2 = np.hstack([te_img.astype(np.float32, copy=False), te_img_missing])



In [4]:
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.exceptions import ConvergenceWarning

gkf = GroupKFold(n_splits=5)
groups = tr["patient_id"].astype(str).values
y = tr["target"].values.astype(int)

# 4 TE + (5 stats + 5 missing flags) = 14 total features
oof_X = np.zeros((len(tr), 14), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(tr, y, groups=groups), 1):
    df_fit = tr.iloc[tr_idx]
    df_val = tr.iloc[va_idx]

    prior_for_maps = global_mean
    fallback_for_apply = global_mean

    sex_rate, site_rate, age_rate, pid_rate = build_te_maps(
        df_fit, prior=prior_for_maps
    )

    X_te_val = apply_te(
        df_val,
        sex_rate,
        site_rate,
        age_rate,
        pid_rate,
        fallback=fallback_for_apply,
        disable_pid=True,
    )

    X_img_val = tr_img2[va_idx].astype(np.float64)
    oof_X[va_idx] = np.hstack([X_te_val.astype(np.float64), X_img_val])

# Change (consistency, minimal): make scaler params explicit for determinism/clarity.
scaler = StandardScaler(with_mean=True, with_std=True)
oof_Xs = scaler.fit_transform(oof_X)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always", ConvergenceWarning)

    blender = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        max_iter=4000,
        C=2.0,
        class_weight="balanced",
        random_state=0,
    )
    blender.fit(oof_Xs, y)

    conv_warnings = [x for x in w if issubclass(x.category, ConvergenceWarning)]
    if len(conv_warnings) > 0:
        print(
            f"ConvergenceWarning count: {len(conv_warnings)} (consider increasing max_iter further)"
        )

oof_pred = blender.predict_proba(oof_Xs)[:, 1]
oof_auc = roc_auc_score(y, oof_pred)
print("OOF ROC-AUC (blender on TE+image features, standardized):", oof_auc)

sex_rate, site_rate, age_rate, pid_rate = build_te_maps(tr, prior=global_mean)

X_te_test = apply_te(
    te, sex_rate, site_rate, age_rate, pid_rate, fallback=global_mean, disable_pid=True
)

X_test = np.hstack([X_te_test.astype(np.float64), te_img2.astype(np.float64)])
X_test_s = scaler.transform(X_test)

pred = blender.predict_proba(X_test_s)[:, 1].astype(float)
pred = np.clip(pred, 0.0, 1.0)

pred_df = pd.DataFrame({"image_name": te["image_name"].values, "target": pred})
sub_out = sub[["image_name"]].merge(pred_df, on="image_name", how="left")
sub_out["target"] = sub_out["target"].fillna(global_mean).astype(float)

sub_out.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", sub_out.shape)
print(sub_out.head())

OOF ROC-AUC (blender on TE+image features, standardized): 0.818994848764753
Wrote submission.csv with shape: (4142, 2)
     image_name    target
0  ISIC_0052212  0.187389
1  ISIC_0076545  0.556133
2  ISIC_0085172  0.082119
3  ISIC_0086709  0.101129
4  ISIC_0109568  0.890362
